# Contract-first single agent for the Gemma 4 Developer Agent competition

Build `submission.zip` for [Google — The Gemma 4 Developer Agent Competition](https://www.kaggle.com/competitions/gemma-4-developer-agent). Run this notebook on **CPU with Internet off**, then submit the `submission.zip` it writes. The notebook does not load Gemma. Kaggle loads the zip, serves `gemma-4-31b-it-qat-w4a16-ct`, and runs the agent on the hidden tasks.

This package keeps the budget that produced the public **0.18** result in [Gemma 4 Agent: Budget-Fit Single Agent](https://www.kaggle.com/code/verracodeguacas/gemma-4-agent-budget-fit-single-agent), and changes the behavior that still turns a solvable issue into a zero. On 6 October 2026 the public leaderboard leader was **0.24**. One public task is worth about 0.017, and identical resubmissions move by about two tasks, so a single score is a noisy measurement.


## 1. The problem

Each task is a real Python issue, not a prediction row.

| Object | What the agent receives | What the agent must not use as an answer |
|---|---|---|
| Issue | `problem_statement` and optional `hints_text` | The reference `patch` |
| Repository | A git snapshot frozen at `base_commit`, with later commits removed | The hidden `test_patch` |
| Navigation aids | Call graphs and 256-d node embeddings, exposed only through harness tools | Memorized fixes for the 129 public practice tasks |

The public practice set has **129 tasks** from `fastapi/fastapi`, `Textualize/rich`, `psf/requests`, and `encode/httpx`, plus snapshots, graphs, embeddings, offline wheels, and a sandbox. The scored set is about **120 tasks from private repositories**, split evenly between the public and private leaderboards. When a submission is scored, the public files are replaced by that hidden set.

The agent works in `/workspace` with no internet. It may read, edit, run a shell command, and call `submit_patch`. The harness turns the working tree into a unified diff. A fresh copy of the baseline receives that diff and the hidden tests. **The task passes only if those tests pass.** There is no partial credit.

```mermaid
flowchart LR
  subgraph attempt [One task, inside the 8 minute cap]
    issue[Issue text] --> agent[Gemma 4 31B agent]
    repo[Frozen repo at /workspace] --> agent
    agent --> diff[git diff]
  end
  diff --> grader[Fresh tree plus hidden tests]
  grader --> score[Pass or fail]
```


## 2. What a legal submission is

The starter notebook (`ryanholbrook/getting-started-gemma-4-developer-agent`) shows the harness path: install the competition wheelhouse, start vLLM, run Phase 1 (the agent writes a patch) and Phase 2 (pytest), then zip the agent directory. Scoring does not use that notebook's two-task smoke run. Scoring uses the zip.

The archive must contain `agent.yaml` at its root. The hosted scorer reads four limits from `eval_config.yaml`: `timeout_seconds`, `max_tool_calls`, `max_time_minutes`, and `max_turns`. Every agent must declare `gemma-4-31b-it-qat-w4a16-ct`. Optional LoRA adapters are allowed. This submission does not include one: a random adapter would change the same base model that the 0.18 prompt was measured with.

Hard limits that this zip respects:

- About 120 tasks must finish inside **12 hours**, including sandbox setup and excluding the grader's own pytest time.
- The model context is **32,768 tokens**. vLLM rejects a request whose prompt plus `max_output_tokens` exceeds that. The starter writes `max_output_tokens: 16384`, which leaves too little room for a long investigation.
- `timeout_seconds` is also the pytest timeout in the verifier. Request test modules need on the order of 100 seconds, and patch extraction has been observed past 120 seconds. A short timeout marks a correct patch as a failure.
- `include_thoughts: false` disables Gemma thinking in the ADK bridge (`enable_thinking: false`). Local comparisons that disabled thinking solved fewer development tasks. `thinking_budget` is accepted by the schema and is not forwarded by the inspected bridge, so it is kept only to match the scoring sampling file.


## 3. What the 0.18 agent already measured

The budget-fit notebook is a single agent on the official sample prompt, with limits taken from local runs rather than from the starter defaults. Those measurements are the floor for this submission. Settings that already scored 0.18 stay put.

| Lever | Starter / common forks | Budget-fit agent (public 0.18) | This submission | Reason |
|---|---|---|---|---|
| Minutes per task | 4, or no cap | 8 | **8** | In 129 local runs, only about half of the fixes Gemma eventually finds are on disk at 4 minutes |
| Tool calls | 40–100 | 28 | **28** | Runs still exploring after many calls almost never succeed |
| Edit deadline | none | edit by call 12 | **edit by call 12** | No source edit by call 12 fails about 87% of the time |
| `max_output_tokens` | 16384 in the starter | 8192 | **8192** | Keeps `prompt + max_output` inside 32,768 |
| `timeout_seconds` | 60–120 | 240 | **240** | The same value bounds the scoring pytest run |
| Thinking | on, in the starter override | on | **on** | Turning it off reduced local solves |
| Graph tools | registered, and the sample prompt recommends them | registered, prompt says not to call them | **not registered** | They require an exact fully qualified symbol id and usually return nothing |
| When to submit | after a targeted test passes | after a targeted test passes | **after the source edit exists** | A red or missing local test is not the hidden grader |

The last two rows are the whole experiment. Same model, same clocks, same sampling. The agent can no longer spend a call on a graph tool that returns empty text, and it can no longer burn the rest of the 8 minutes waiting for a local test that the repository does not have.


## 4. The repair loop

The hidden tests check exact names, messages, and status codes. A nearly correct patch scores the same as no patch. The prompt therefore copies the contract out of the issue before it searches, then forces a small edit while calls remain for one correction.

```mermaid
flowchart TD
  A[Issue text] --> B[Copy the contract verbatim]
  B --> C["git grep, then read at most 120 lines"]
  C --> D{Source edited by call 12?}
  D -->|Not yet| E[Edit the most likely function now]
  D -->|Yes| F[At most one focused check]
  E --> F
  F --> G{Assertion about this edit?}
  G -->|Yes| H[One repair, then stop testing]
  G -->|No: fixture, import, or pre-existing failure| I[Ignore it]
  H --> J[Remove test paths and scratch files from the diff]
  I --> J
  J --> K[submit_patch]
  K --> L[Grader: fresh tree, hidden pytest, pass or fail]
```

Scratch files belong in `/tmp`. `submit_patch` records untracked files under `/workspace`, so a reproduction script left in the repo becomes part of the graded diff. Test edits are restored unreliably by the harness and fail the task when they stick. The prompt checks the diff and checks those paths out before the final submit.

Wall-clock arithmetic: 120 tasks times 8 minutes is 16 hours, which would miss the 12-hour cap if every task used the whole cap. The cap is a maximum. The budget-fit author reported a scored run under these caps finished inside 12 hours, because most tasks stop early. Submitting as soon as the edit is on disk makes the average shorter than an agent that keeps testing until the clock ends.


In [ ]:
from pathlib import Path

def workflow_svg() -> str:
    """Stdlib flowchart of one graded attempt. No extra packages."""
    def box(x, y, w, h, title, body, fill):
        return (
            f'<rect x="{x}" y="{y}" width="{w}" height="{h}" rx="12" '
            f'fill="{fill}" stroke="#1c1917" stroke-width="1.6"/>'
            f'<text x="{x + 18}" y="{y + 32}" font-family="Georgia, serif" '
            f'font-size="16" font-weight="700" fill="#1c1917">{title}</text>'
            f'<text x="{x + 18}" y="{y + 56}" font-family="ui-sans-serif, sans-serif" '
            f'font-size="13" fill="#44403c">{body}</text>'
        )

    def arrow(x, y1, y2):
        return (
            f'<line x1="{x}" y1="{y1}" x2="{x}" y2="{y2}" stroke="#1c1917" '
            f'stroke-width="1.6" marker-end="url(#arrow)"/>'
        )

    steps = [
        (78, "1  Contract", "Copy exact names, messages, and codes", "#ecfdf5"),
        (176, "2  Locate", "One git grep, then at most 120 lines", "#f5f5f4"),
        (274, "3  Edit by call 12", "Smallest source change. Stop reading.", "#fff7ed"),
        (372, "4  One check", "Single pytest -k, or skip a broken suite", "#f5f5f4"),
        (470, "5  One repair", "Only if the assertion is about this edit", "#fff7ed"),
        (568, "6  submit_patch", "Drop tests and scratch files, then stop", "#ecfdf5"),
        (666, "7  Hidden grader", "Fresh tree. Whole task passes or fails.", "#fee2e2"),
    ]
    x, w, h = 48, 430, 78
    parts = [
        '<svg xmlns="http://www.w3.org/2000/svg" width="1000" height="790" viewBox="0 0 1000 790">',
        '<defs><marker id="arrow" markerWidth="8" markerHeight="8" refX="7" refY="3" orient="auto">',
        '<path d="M0,0 L7,3 L0,6 Z" fill="#1c1917"/></marker></defs>',
        '<rect width="1000" height="790" fill="#fafaf9"/>',
        '<text x="48" y="42" font-family="Georgia, serif" font-size="24" fill="#1c1917">One hidden task</text>',
        '<text x="48" y="64" font-family="ui-sans-serif, sans-serif" font-size="13" fill="#57534e">The agent loop is on the left. The grader never shows its tests to the agent.</text>',
    ]
    for i, (y, title, body, fill) in enumerate(steps):
        parts.append(box(x, y, w, h, title, body, fill))
        if i:
            parts.append(arrow(x + w / 2, steps[i - 1][0] + h, y - 8))
    parts.append(
        '<rect x="520" y="78" width="412" height="268" rx="12" fill="#ffffff" stroke="#1c1917" stroke-width="1.6"/>'
        '<text x="544" y="114" font-family="Georgia, serif" font-size="18" fill="#1c1917">Budget kept from 0.18</text>'
        '<text x="544" y="154" font-family="ui-sans-serif, sans-serif" font-size="15" fill="#1c1917">8 minutes per task</text>'
        '<text x="544" y="182" font-family="ui-sans-serif, sans-serif" font-size="15" fill="#1c1917">28 tool calls</text>'
        '<text x="544" y="210" font-family="ui-sans-serif, sans-serif" font-size="15" fill="#1c1917">240s command and pytest cap</text>'
        '<text x="544" y="238" font-family="ui-sans-serif, sans-serif" font-size="15" fill="#1c1917">8192 output tokens, 32768 context</text>'
        '<text x="544" y="266" font-family="ui-sans-serif, sans-serif" font-size="15" fill="#1c1917">Thinking stays on</text>'
        '<text x="544" y="308" font-family="ui-sans-serif, sans-serif" font-size="13" fill="#57534e">Graph tools are not registered.</text>'
    )
    parts.append(
        '<rect x="520" y="372" width="412" height="236" rx="12" fill="#ffffff" stroke="#1c1917" stroke-width="1.6"/>'
        '<text x="544" y="408" font-family="Georgia, serif" font-size="18" fill="#1c1917">What the grader counts</text>'
        '<text x="544" y="448" font-family="ui-sans-serif, sans-serif" font-size="15" fill="#1c1917">Hidden tests all pass</text>'
        '<text x="544" y="476" font-family="ui-sans-serif, sans-serif" font-size="15" fill="#1c1917">Strings copied from the issue</text>'
        '<text x="544" y="504" font-family="ui-sans-serif, sans-serif" font-size="15" fill="#1c1917">No test files in the diff</text>'
        '<text x="544" y="532" font-family="ui-sans-serif, sans-serif" font-size="15" fill="#1c1917">An empty diff scores zero</text>'
        '<text x="544" y="574" font-family="ui-sans-serif, sans-serif" font-size="13" fill="#57534e">One public task is about 0.017.</text>'
    )
    parts.append("</svg>")
    return "".join(parts)


svg = workflow_svg()
out = Path("/kaggle/working") if Path("/kaggle/working").is_dir() else Path.cwd()
(out / "workflow.svg").write_text(svg, encoding="utf-8")
try:
    from IPython.display import SVG, display
    display(SVG(data=svg))
except Exception:
    print(f"Wrote {out / 'workflow.svg'}")


## 5. What this can and cannot claim

A public score of 0.18 is about 10 solved tasks out of roughly 58. A 0.24 is about 14. The changes above are aimed at tasks the 0.18 agent already understood well enough to edit, then lost: a wasted graph call, a contract string that was paraphrased, a diff polluted by a test or a scratch file, or a session that died while hunting a green local pytest.

That is a plausible route from 0.18 to the current public leader. It is not a measured score. The private half of the test set is a different 50%, and two identical submissions can differ by a couple of tasks. Run this once as the daily submission. Do not treat a one-task move as a new design.

Deliberately left out, because local evidence said they do not pay for their cost:

- A second agent. A read-only analyzer beside a coder solved fewer development tasks than the coder alone.
- A longer cap than 8 minutes. The queue is shared. Minutes spent on a task that will still fail are taken from a later task.
- A LoRA trained on the 129 public patches. Those repositories are not the hidden set, and the zip would no longer be the agent that the budget measurements describe.


In [ ]:
import hashlib
import zipfile
from pathlib import Path

FILES = {
    "agent.yaml": """name: swe_agent
model: gemma-4-31b-it-qat-w4a16-ct
instruction: !include prompts/system.md
tools:
  - run_command
  - read_file
  - edit_file
  - write_file
  - get_status
  - submit_patch
generate_content_config: !include configs/sampling.yaml
""",
    "configs/sampling.yaml": """temperature: 0.2
top_p: 0.95
max_output_tokens: 8192
thinking_config:
  thinking_budget: 4096
  include_thoughts: true
""",
    "eval_config.yaml": """evaluation:
  timeout_seconds: 240
  max_tool_calls: 28
  max_time_minutes: 8
  max_turns: 80
""",
    "prompts/system.md": """You are an expert autonomous software engineer. Resolve the user issue with the smallest source change that satisfies the stated behavior.

The hidden grader applies your git diff to a fresh repository and runs its own tests. It never sees your reasoning. An empty diff scores zero. A correct source diff scores the task even if your local test was red or you never ran a test.

## Budget
You have 28 tool calls and 8 minutes. When either limit hits, the working tree is graded as it is. An edit made in time counts even if you do not reach submit_patch. Finish as soon as the edit is in so later tasks still have time.

## 1. Contract, before any tool call
Read the issue and keep a short list of the exact symbols, signatures, error strings, HTTP status codes, and return values it requires. Copy them character for character. The grader checks those strings exactly.

## 2. Locate in a few reads
- Search once with `git grep -n "SPECIFIC_TOKEN" -- '*.py' | head -20`. Use the most specific identifier or error string from the issue.
- Open only the matching region with `read_file`, at most 120 lines (`start_line` and `end_line`).
- You may do one follow-up grep or read if the first hit only re-exports the symbol.
- Do not page through unrelated files. Do not read a whole file when a range will do.

## 3. Edit by tool call 12
- By your 12th tool call a source file must already be edited. If it is not, edit the most likely function immediately. Runs that are still only reading after 12 calls almost never succeed.
- Prefer `edit_file`. `old_string` must match the file exactly, including indentation. Change the smallest span that implements the contract.
- FastAPI documentation tasks: edit the executable example under `docs_src/`.
- Do not reformat or refactor unrelated code.
- Create a new file only when the issue explicitly requires a new module. Never create or edit `test_*.py`, `*_test.py`, anything under `tests/`, `pytest.ini`, or `conftest.py`.
- Any scratch script goes in `/tmp` through `run_command`. Never write scratch files into `/workspace`. Files left in the repository become part of the diff and fail grading.

## 4. One check, one repair, then submit
- After the edit, run at most one focused check: `pytest tests/test_file.py -k test_name -q` or `python -c "..."`.
- Never run bare `pytest`, `pytest .`, or `python -m unittest discover`.
- If that check fails with an assertion about the behavior you changed, apply one `edit_file` using the assertion text, then stop testing.
- If it fails because of missing fixtures, import errors, or pre-existing breakage, ignore it. Do not repair tests and do not install packages.
- Call `submit_patch` after that, even when the local check is red or you skipped it. A local green test is not the grader.
- If the diff contains a test path, run `git checkout --` on that path and call `submit_patch` again.
- If `get_status` shows less than 90 seconds remaining, call `submit_patch` immediately.

## Never
- Conclude that the code is already correct. Every task needs a source modification and a non-empty patch.
- Search outside `/workspace` (`/usr`, `/wheels`, `/opt`). Dependencies are already installed.
- Spend calls on broad exploration after the first edit.
""",
}

EXPECTED = {
    "agent.yaml",
    "configs/sampling.yaml",
    "eval_config.yaml",
    "prompts/system.md",
}
ALLOWED_EXT = {".yaml", ".yml", ".md", ".json", ".safetensors", ".txt", ".py"}
REQUIRED_TOOLS = [
    "run_command",
    "read_file",
    "edit_file",
    "write_file",
    "get_status",
    "submit_patch",
]
FORBIDDEN_TOOLS = ["get_code_neighbors", "search_similar_code", "get_code_subgraph"]


def check_archive(files: dict) -> None:
    if set(files) != EXPECTED:
        raise SystemExit(f"Unexpected archive contents: {sorted(files)}")
    agent = files["agent.yaml"]
    if "model: gemma-4-31b-it-qat-w4a16-ct" not in agent:
        raise SystemExit("The only permitted base model is missing.")
    if "instruction: !include prompts/system.md" not in agent:
        raise SystemExit("agent.yaml must include prompts/system.md.")
    if "generate_content_config: !include configs/sampling.yaml" not in agent:
        raise SystemExit("agent.yaml must include configs/sampling.yaml.")
    for tool in REQUIRED_TOOLS:
        if f"- {tool}\n" not in agent:
            raise SystemExit(f"Missing tool: {tool}")
    for tool in FORBIDDEN_TOOLS:
        if tool in agent:
            raise SystemExit(f"Graph tool still registered: {tool}")
    sampling = files["configs/sampling.yaml"]
    for needle in ("temperature: 0.2\n", "top_p: 0.95\n", "max_output_tokens: 8192\n", "include_thoughts: true\n"):
        if needle not in sampling:
            raise SystemExit(f"Sampling drift: {needle!r}")
    evaluation = files["eval_config.yaml"]
    for needle in (
        "timeout_seconds: 240\n",
        "max_tool_calls: 28\n",
        "max_time_minutes: 8\n",
        "max_turns: 80\n",
    ):
        if needle not in evaluation:
            raise SystemExit(f"Budget drift: {needle!r}")
    prompt = files["prompts/system.md"]
    for needle in (
        "character for character",
        "12th tool call",
        "submit_patch",
        "/tmp",
        "docs_src/",
        "local green test is not the grader",
        "less than 90 seconds",
    ):
        if needle not in prompt:
            raise SystemExit(f"Prompt is missing the rule: {needle}")
    if "search_similar_code" in prompt or "get_code_neighbors" in prompt:
        raise SystemExit("Prompt still mentions a graph tool by name.")
    unpacked = sum(len(text.encode()) for text in files.values())
    if unpacked >= 3 * 1024**3:
        raise SystemExit("Unpacked submission must stay under 3 GiB.")


check_archive(FILES)

out = Path("/kaggle/working") if Path("/kaggle/working").is_dir() else Path.cwd()
archive = out / "submission.zip"
with zipfile.ZipFile(archive, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    for name in sorted(FILES):
        ext = Path(name).suffix.lower()
        if ext not in ALLOWED_EXT:
            raise SystemExit(f"Disallowed extension: {name}")
        zf.writestr(name, FILES[name])

with zipfile.ZipFile(archive) as zf:
    names = set(zf.namelist())
    if zf.testzip() is not None or names != EXPECTED or "agent.yaml" not in names:
        raise SystemExit("Archive verification failed.")
    bad_root = [n for n in names if n.startswith("/") or n.startswith("..") or n.split("/")[0] == "agent"]
    if bad_root:
        raise SystemExit(f"agent.yaml must sit at the archive root, found {bad_root}")

digest = hashlib.sha256(archive.read_bytes()).hexdigest()
print(f"Wrote {archive} ({archive.stat().st_size} bytes)")
print(f"SHA256 {digest}")
print("Files:")
for name in sorted(FILES):
    print(f"  {name:28} {len(FILES[name].encode()):5} bytes")
print("Budget: 8 min, 28 tool calls, 240 s command/pytest timeout, 80 turns")
print("Model:  gemma-4-31b-it-qat-w4a16-ct")
print("Submit this notebook output. Kaggle scores submission.zip, not this cell's printout.")


## 6. How to submit

1. Copy this notebook into the competition, or upload it as a competition notebook.
2. Accelerator: **None** (CPU). Internet: **off**.
3. Run All. Confirm the cell prints `agent.yaml` at the archive root and a SHA-256.
4. Submit the notebook. Kaggle scores `/kaggle/working/submission.zip`.
5. The contest allows one scored submission a day. Use this zip for that submission.

The printout is a fingerprint of the archive. Keep it next to the public score so a later change is a different experiment, not a silent edit of this one.
